In [57]:
import torch
from torch import nn
from d2l import torch as d2l

In [58]:
def gaussian(
    x: torch.Tensor,
) -> torch.Tensor:
    return torch.exp(
        -(x**2) / 2
    )


def boxcar(
    x: torch.Tensor,
) -> torch.Tensor:
    return (
        torch.abs(x) < 1.0
    ).to(dtype=torch.float32)


def constant(
    x: torch.Tensor,
) -> torch.Tensor:
    return torch.ones_like(x)


def epanechikov(
    x: torch.Tensor,
) -> torch.Tensor:
    return torch.maximum(
        1 - torch.abs(x),
        torch.zeros_like(x),
    )


def target_function(
    x: torch.Tensor,
) -> torch.Tensor:
    return (
        2 * torch.sin(x)
        + x
    )



kernels = (
    gaussian,
    boxcar,
    constant,
    epanechikov,
)

kernel_names = (
    "Gaussian",
    "Boxcar",
    "Constant",
    "Epanechikov",
)


num_train = 40


# Key:
x_train, _ = torch.sort(
    torch.rand(num_train) * 5
)

# Value:
y_train = (
    target_function(x_train)
    + torch.randn(num_train)
)


# Query: prediction을 구할 위치
x_val = torch.arange(
    0,
    5,
    0.1,
)

y_val = target_function(
    x_val
)


print(
    "Number of Keys:",
    x_train.numel(),
)
print(
    "Number of Queries:",
    x_val.numel(),
)

# Keys    : x_train [40]
# Values  : y_train [40]
# Queries : x_val   [50]

Number of Keys: 40
Number of Queries: 50


In [59]:
# Nadaraya-Watson Attention Pooling

def nadaraya_watson(
    x_train: torch.Tensor, # Key  
    y_train: torch.Tensor, # Value 
    queries: torch.Tensor, # Query
    kernel,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
]:
    
    # Distances k ~ q:
    # Key:[K, 1] - Query:[1, Q] -> [K, Q]
    distances = (
        x_train.reshape(-1, 1)   # x_train: [K, 1]
        - queries.reshape(1, -1) # queries: [1, Q]
    )
    
    # Transformation: distances -> similarities
    similarities = kernel(
        distances
    ).to(dtype=torch.float32)
    
    # Attention Weights / Normalization to each Query:
    # [K, Q] / [1, Q] -> [K, Q]
    attention_weights = (
        similarities
        / similarities.sum(
            dim=0,
            keepdim=True,
        )
    )
    
    # 각 Query에 대해 Prediction:
    # Value:[K] @ Weight:[K, Q] -> [Q]
    predictions = (
        y_train @ attention_weights
    )
    
    return (
        predictions,
        attention_weights,
    )

In [60]:
# Gaussian Attention 추적

# [K, Q]
distances = (
    x_train.reshape(-1, 1)   # x_train: [K, 1]
    - x_val.reshape(1, -1)   # queries: [1, Q]
)
    
# [K, Q]
similarities = gaussian(
    distances
).to(dtype=torch.float32)


# [Q], [K, Q]
predictions, attention_weights = (
    nadaraya_watson(
        x_train=x_train,
        y_train=y_train,
        queries=x_val,
        kernel=gaussian,
    )
)

print(
    "distances shape:",
    tuple(distances.shape),
)

print(
    "similarities shape:",
    tuple(similarities.shape),
)

print(
    "Attention Weight shape:",
    tuple(attention_weights.shape),
)
print(
    "Prediction shape:",
    tuple(predictions.shape),
)

# 각 Query에 대한 40개 Key weight의 합은 1
weight_sums = attention_weights.sum(
    dim=0,
)

print(
    "First five Weight sums:",
    weight_sums[:5],
)


distances shape: (40, 50)
similarities shape: (40, 50)
Attention Weight shape: (40, 50)
Prediction shape: (50,)
First five Weight sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
